## TON IoT Data

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# import pandas as pd
# import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report, precision_recall_fscore_support
import matplotlib.pyplot as plt
# import torch.nn as nn
# import torch.nn.functional as F
# import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import math, time, os, datetime, psutil, gc
import random
import warnings

%load_ext autoreload
%autoreload 2
from utility import *
from informer import *
from libs import *

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Ingestion Step

In [2]:
os.listdir('../data/TON_IoT/')

['train_test_network.csv',
 'IoT_Garage_Door.csv',
 'IoT_Thermostat.csv',
 'Train_Test_IoT_Fridge.xlsx',
 'IoT_GPS_Tracker.csv',
 'IoT_Fridge.csv',
 'df_iot_os_linux.pkl',
 'IoT_Modbus.csv',
 'IoT_Weather.csv',
 'IoT_Motion_Light.csv']

In [3]:
file = '../data/TON_IoT/IoT_Weather.csv'
sensor_name = "Weather"
file_name = "classification_report.xlsx"

In [4]:
### IOT Network
# df= pd.read_excel(file)
df= pd.read_csv(file)
df.head()
col_resp='type'

,date,time,temperature,pressure,humidity,label,type
0,31-Mar-19,12:36:52,31.788508,1.035,32.036579,0,normal
1,31-Mar-19,12:36:53,41.630997,1.035,30.886165,0,normal
2,31-Mar-19,12:36:54,42.256959,1.035,19.755908,0,normal
3,31-Mar-19,12:36:55,49.116581,1.035,78.949621,0,normal
4,31-Mar-19,12:36:56,24.017085,1.035,40.001059,0,normal


In [5]:
df=df.dropna(subset=['date','time'])

In [6]:
# Prepare data
df.replace('-', 'Missing', inplace=True)
df.fillna('Missing', inplace=True)

# --- Clean date & time ---
df["date"] = df["date"].astype(str).str.strip()
df["time"] = df["time"].astype(str).str.strip()

# --- Create datetime ---
df["datetime"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    format="%d-%b-%y %H:%M:%S",
    # format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# --- Clean response column ---
df[col_resp] = df[col_resp].astype(str).str.strip()

# --- Drop old columns ---
df.drop(columns=['date', 'time'], inplace=True)

# --- Extract features ---
dt = df["datetime"]

df["month"]   = dt.dt.month
df["day"]     = dt.dt.day
df["weekday"] = dt.dt.weekday
df["hour"]    = dt.dt.hour
df["minute"]  = dt.dt.minute
df["second"]  = dt.dt.second

df = df.sort_values('datetime').reset_index(drop=True)

df.drop(columns=['datetime'], inplace=True)

time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']

In [7]:
df.pivot_table(index=['month', 'day'], columns='type', aggfunc='size', fill_value=0)

type       backdoor   ddos  injection  normal  password  ransomware  scanning  \
month day                                                                       
3     31          0      0          0   46348         0           0         0   
4     1           0      0          0   52090         0           0         0   
      2           0      0          0  243230         0           0         0   
      3           0      0          0   86268         0           0         0   
      4           0      0          0   28200         0           0         0   
      17          0      0          0    1057         0           0         0   
      23          0      0          0    5908         0           0       529   
      24          0      0          0    2656         0           0         0   
      25          0  13499       9726    7250         0           0         0   
      26          0   1683          0   22335     17216           0         0   
      27          0      0          0   45024      8499           0         0   
      28      30152      0          0    1519         0        2865         0   
      29       5489      0          0   17833         0           0         0   

type       xss  
month day       
3     31     0  
4     1      0  
      2      0  
      3      0  
      4      0  
      17     0  
      23     0  
      24     0  
      25     0  
      26     0  
      27   866  
      28     0  
      29     0

In [8]:
display(df)

,temperature,pressure,humidity,label,type,month,day,weekday,hour,minute,second
0,31.788508,1.035000,32.036579,0,normal,3,31,6,12,36,52
1,31.788508,1.035000,32.036579,0,normal,3,31,6,12,36,52
2,41.630997,1.035000,30.886165,0,normal,3,31,6,12,36,53
3,41.630997,1.035000,30.886165,0,normal,3,31,6,12,36,53
4,42.256959,1.035000,19.755908,0,normal,3,31,6,12,36,54
...,...,...,...,...,...,...,...,...,...,...,...
650237,44.231998,2.495434,95.049929,0,normal,4,29,0,14,47,31
650238,36.771845,1.155912,25.337601,0,normal,4,29,0,14,47,33
650239,47.943151,1.783286,48.870661,0,normal,4,29,0,14,47,35
650240,44.573469,2.495434,95.049929,0,normal,4,29,0,14,47,36


In [9]:
# 1. Prepare Features and Target
X = df.drop(columns=['label', 'type'])
y = df[col_resp].str.lower()  # Force lowercase to prevent 'Normal' vs 'normal' mismatch

# 2. Force "normal" to 0, others to 1...N
# Get unique classes and move 'normal' to the front
unique_classes = np.unique(y)
other_classes = [c for c in unique_classes if c != 'normal']
class_names = ['normal'] + sorted(other_classes)

# Create the mapping dictionary
label_map = {name: i for i, name in enumerate(class_names)}
print(f"Manual Mapping: {label_map}")

# Apply the mapping to y
y_encoded = y.map(label_map).values

# 3. Encode categorical features in X
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype(str).str.lower() # Consistency for features too
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

print(f"Encoded classes: {class_names}")
# categorical_cols = ['thermostat_status']
# X[categorical_cols] = X[categorical_cols].astype('category')

Manual Mapping: {'normal': 0, 'backdoor': 1, 'ddos': 2, 'injection': 3, 'password': 4, 'ransomware': 5, 'scanning': 6, 'xss': 7}
Encoded classes: ['normal', 'backdoor', 'ddos', 'injection', 'password', 'ransomware', 'scanning', 'xss']


In [10]:
from sklearn.model_selection import train_test_split

# 1. First split: 80% for Training+Validation, 20% for Final Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)

# 2. Second split: From the 80%, take 15% for Validation (leaving 68% total for Train)
# Adjust test_size=0.1875 because 0.1875 * 0.8 = 0.15 of the total dataset
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=20, stratify=y_temp
)

# 3. Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)      # Use transform, NOT fit_transform
X_test_scaled = scaler.transform(X_test)    # Use transform, NOT fit_transform

# 4. Create Tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train).flatten()

X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val).flatten()

X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test).flatten()

# 5. Prepare DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print(f"Train size: {len(X_train_tensor)} | Val size: {len(X_val_tensor)} | Test size: {len(X_test_tensor)}")

Train size: 312116 | Val size: 78029 | Test size: 260097


Performance measurement Preparation

In [11]:
import psutil
import os
import pynvml
import time

# Initialize system process
process = psutil.Process(os.getpid())

# Initialize GPU handle if available
if torch.cuda.is_available():
    try:
        pynvml.nvmlInit()
        gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_handle = None

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "DCNN" 

# ==========================================
# 4. Build Deep Learning Model
# ==========================================
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) 
            # Note: No Softmax here because CrossEntropyLoss includes it
        )

    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
num_classes = len(class_names)
model = DeepClassifier(input_dim, num_classes).to(device)

# Define Optimizer and Loss
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss() # Equivalent to sparse_categorical_crossentropy

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=2, 
    factor=0.5, 
    verbose=True
)

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [13]:
import time

# ... (Model, Optimizer, and Criterion initialization) ...

print(f"Training started on {device}...")
print("-" * 50)

# Computational Measurement Start
start_stats_train = get_system_stats(device, reset=True)
start_time_train = time.time()

for epoch in range(20):
    _=model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        _=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # .item() is key here: it converts a 1-element tensor to a standard Python float
        train_loss += loss.item() 

    # --- VALIDATION PHASE ---
    _=model.eval()
    val_loss, v_preds, v_true, v_probs = 0.0, [], [], []
    
    with torch.no_grad():
        for v_X, v_y in val_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            
            val_loss += criterion(v_out, v_y).item()
            
            v_probs.extend(torch.softmax(v_out, dim=1).cpu().numpy())
            v_preds.extend(torch.max(v_out, 1)[1].cpu().numpy())
            v_true.extend(v_y.cpu().numpy())

    # --- METRICS & LOGGING ---
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    acc = accuracy_score(v_true, v_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(v_true, v_preds, average='weighted', zero_division=0)
    auroc = roc_auc_score(v_true, v_probs, multi_class='ovr', average='weighted')

    # This single print statement handles the entire epoch output
    print(f"Epoch [{epoch+1:02d}/20] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"STATS >> Acc: {acc:.2f}% | F1: {f1:.4f} | AUROC: {auroc:.4f}")
    print("-" * 50)

print("Training Complete!")

## Training Computational Performance End
train_duration = time.time() - start_time_train
end_stats_train = get_system_stats(device)


Training started on cuda...
--------------------------------------------------
Epoch [01/20] | Train Loss: 0.1020 | Val Loss: 0.0478
STATS >> Acc: 98.23% | F1: 0.9820 | AUROC: 0.9975
--------------------------------------------------
Epoch [02/20] | Train Loss: 0.0453 | Val Loss: 0.0329
STATS >> Acc: 98.79% | F1: 0.9874 | AUROC: 0.9986
--------------------------------------------------
Epoch [03/20] | Train Loss: 0.0390 | Val Loss: 0.0298
STATS >> Acc: 98.95% | F1: 0.9893 | AUROC: 0.9989
--------------------------------------------------
Epoch [04/20] | Train Loss: 0.0350 | Val Loss: 0.0279
STATS >> Acc: 99.04% | F1: 0.9904 | AUROC: 0.9989
--------------------------------------------------
Epoch [05/20] | Train Loss: 0.0324 | Val Loss: 0.0254
STATS >> Acc: 99.12% | F1: 0.9912 | AUROC: 0.9991
--------------------------------------------------
Epoch [06/20] | Train Loss: 0.0304 | Val Loss: 0.0252
STATS >> Acc: 99.22% | F1: 0.9923 | AUROC: 0.9991
------------------------------------------

In [14]:
# 1. Generate Training Classification report
df_metrics_train = get_classification_df(v_true, 
                                         v_preds,
                                         v_probs, 
                                         class_names=class_names,
                                         sensor_name=sensor_name, 
                                         model_name=model_name, 
                                         phase="Training")

#1. Generate computational performance report
df_perf_train = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Training',
    duration=train_duration,
    stats=end_stats_train
)

In [15]:
# ==========================================
# 6. Final Evaluation (Testing Phase)
# ==========================================
print("\n" + "="*50)
print("FINAL TEST EVALUATION STARTED")
print("="*50)

# 1. Reset Measurements for Testing Phase
start_stats_test = get_system_stats(device, reset=True)
start_time_test = time.time()

# 2. Run Inference
_=model.eval()
test_loss, t_preds, t_true, t_probs = 0.0, [], [], []

with torch.no_grad():
    for t_X, t_y in test_loader:
        t_X, t_y = t_X.to(device), t_y.to(device)
        t_out = model(t_X)
        
        test_loss += criterion(t_out, t_y).item()
        
        # Collect probabilities and predictions
        t_probs.extend(torch.softmax(t_out, dim=1).cpu().numpy())
        t_preds.extend(torch.max(t_out, 1)[1].cpu().numpy())
        t_true.extend(t_y.cpu().numpy())

# 3. End Measurements
test_duration = time.time() - start_time_test
end_stats_test = get_system_stats(device)


FINAL TEST EVALUATION STARTED


In [16]:
# 1. Generate Testing Classification report
df_metrics_test = get_classification_df(
    y_true=t_true, 
    y_pred=t_preds, 
    y_probs=t_probs,
    class_names=class_names,
    sensor_name=sensor_name, 
    model_name=model_name, 
    phase="Testing"
)

# 2. Generate Testing Computational performance report
df_perf_test = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Testing',
    duration=test_duration,
    stats=end_stats_test
)

In [17]:
# Final Reports
# Classification
df_metrics = pd.concat([df_metrics_train, df_metrics_test], axis=0)
display(df_metrics)

# Classification
df_perf = pd.concat([df_perf_train, df_perf_test], axis=0)
display(df_perf)

#Saving Classification Report
save_report(df=df_metrics, report_type="classification",output_dir="../output/")

save_report(df=df_perf, report_type="performance",output_dir="../output/")

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.999043,0.994834,0.996934,67166.000000,DCNN,Weather,Training
1,backdoor,0.967969,0.996259,0.981910,4277.000000,DCNN,Weather,Training
2,ddos,0.932855,0.998902,0.964750,1822.000000,DCNN,Weather,Training
3,injection,0.994881,0.999143,0.997007,1167.000000,DCNN,Weather,Training
4,password,0.984868,0.991251,0.988049,3086.000000,DCNN,Weather,Training
5,ransomware,0.945055,1.000000,0.971751,344.000000,DCNN,Weather,Training
6,scanning,0.980000,0.777778,0.867257,63.000000,DCNN,Weather,Training
7,xss,0.990099,0.961538,0.975610,104.000000,DCNN,Weather,Training
8,accuracy,0.994733,0.994733,0.994733,0.994733,DCNN,Weather,Training
9,macro avg,0.974346,0.964963,0.967909,78029.000000,DCNN,Weather,Training


,Metric,Value,Unit,model,sensor,phase
0,Duration,86.0752,Seconds,DCNN,Weather,Training
1,Peak CPU Memory,1360.4700,MB,DCNN,Weather,Training
2,Avg CPU Utilization,7.8000,%,DCNN,Weather,Training
3,Peak GPU Memory,18.0300,MB,DCNN,Weather,Training
4,Avg GPU Utilization,0.0000,%,DCNN,Weather,Training
0,Duration,1.1935,Seconds,DCNN,Weather,Testing
1,Peak CPU Memory,1413.5200,MB,DCNN,Weather,Testing
2,Avg CPU Utilization,7.4000,%,DCNN,Weather,Testing
3,Peak GPU Memory,18.0300,MB,DCNN,Weather,Testing
4,Avg GPU Utilization,0.0000,%,DCNN,Weather,Testing


✅ Report successfully saved to: ../output/classification_report.xlsx | Sheet: DCNN_Weather
✅ Report successfully saved to: ../output/computational_performance.xlsx | Sheet: DCNN_Weather


# Informer

In [18]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df[col_resp] = df[col_resp].astype(str).str.strip()
# df[col_resp] = le.fit_transform(df[col_resp])
# df[col_resp].value_counts()

if(len(np.array(df[col_resp].value_counts()))>2):
    # 1. Get unique types and remove 'normal'
    attack_types = [t for t in df[col_resp].unique() if t != 'normal']

    # 2. Create a mapping where 'normal' is always 0
    mapping = {t: i+1 for i, t in enumerate(attack_types)}
    mapping['normal'] = 0

    # 3. Apply the mapping
    df[col_resp] = df[col_resp].map(mapping)

    print(f"Manual Mapping: {mapping}")

Manual Mapping: {'scanning': 1, 'injection': 2, 'ddos': 3, 'password': 4, 'xss': 5, 'ransomware': 6, 'backdoor': 7, 'normal': 0}


## Data Preprocessing

In [19]:
df_new = X
df_new['type'] = y_encoded
df_new

,temperature,pressure,humidity,month,day,weekday,hour,minute,second,type
0,31.788508,1.035000,32.036579,3,31,6,12,36,52,0
1,31.788508,1.035000,32.036579,3,31,6,12,36,52,0
2,41.630997,1.035000,30.886165,3,31,6,12,36,53,0
3,41.630997,1.035000,30.886165,3,31,6,12,36,53,0
4,42.256959,1.035000,19.755908,3,31,6,12,36,54,0
...,...,...,...,...,...,...,...,...,...,...
650237,44.231998,2.495434,95.049929,4,29,0,14,47,31,0
650238,36.771845,1.155912,25.337601,4,29,0,14,47,33,0
650239,47.943151,1.783286,48.870661,4,29,0,14,47,35,0
650240,44.573469,2.495434,95.049929,4,29,0,14,47,36,0


## Train Test Split

In [20]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader

# --- A. Define Features & Create Sequences ---
feature_cols = [col for col in df_new.columns if col not in 
                ['label', 'type']]


In [21]:
## Setting Parameters
# data loading params
seq_len = 360
batch_size = 32

# Model training params
pred_len = 1
d_ff=512
d_model=512
num_epochs = 50

In [22]:
time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']
cat_cols = list(categorical_cols)
num_cols = [col for col in feature_cols if col not in cat_cols+time_cols]
all_features = num_cols + cat_cols + time_cols

In [23]:
# # --- STEP 0: Feature Definitions ---
# time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']
# cat_cols = categorical_cols
# num_cols = [col for col in feature_cols if col not in cat_cols+time_cols]
# all_features = num_cols + cat_cols + time_cols

# Map for the model to "find" indices and define embedding sizes
config = {
    'num_idx': [all_features.index(c) for c in num_cols],
    'cat_idx': [all_features.index(c) for c in cat_cols],
    'time_idx': [all_features.index(c) for c in time_cols],
    # Get unique counts for embeddings (+1 for safety/padding)
    'cat_dims': [df_new[c].nunique() + 1 for c in cat_cols],
    'time_dims': {
        'month': 13, 'day': 32, 'weekday': 7, 
        'hour': 24, 'minute': 60, 'second': 60
    }
}

In [24]:
def create_sequences_fixed(df, feature_list, seq_len=64, y_col='type'):
    X, y = [], []
    
    # Get values for all features
    data = df[feature_list].values
    labels = df[y_col].values
    
    # Slide across the whole dataset
    step = seq_len // 2 
    for i in range(0, len(data) - seq_len + 1, step):
        X.append(data[i : i + seq_len])
        # Get label of the last timestamp in the window
        y.append(labels[i + seq_len - 1])
                
    return np.array(X), np.array(y)

In [25]:
# 1. Split IDs and create DataFrames (3-way split)
ids = df_new.index
# First split: 60% Train+Val, 40% Test
train_val_ids, test_ids = train_test_split(ids, test_size=0.4, random_state=42, shuffle=True)
# Second split: From that 60%, take 20% for Validation (leaving 48% total for Train)
train_ids, val_ids = train_test_split(train_val_ids, test_size=0.2, random_state=42, shuffle=True)

df_train = df_new[df_new.index.isin(train_ids)]
df_val = df_new[df_new.index.isin(val_ids)]
df_test = df_new[df_new.index.isin(test_ids)]

# 2. Create Sequences
X_train_seq, y_train_seq = create_sequences_fixed(df_train, all_features, seq_len=64)
X_val_seq, y_val_seq = create_sequences_fixed(df_val, all_features, seq_len=64)
X_test_seq, y_test_seq = create_sequences_fixed(df_test, all_features, seq_len=64)

# 3. SMOTE (Apply ONLY to Training data)
n_samples, seq_len, n_features = X_train_seq.shape
X_train_flat = X_train_seq.reshape(n_samples, -1)

# Check the minimum number of samples in any class
min_samples = np.min(np.bincount(y_train_seq))

if min_samples > 1:
    # Set k_neighbors to min(5, min_samples - 1)
    k = min(5, min_samples - 1)
    sm = SMOTE(random_state=20, k_neighbors=k)
    X_res_flat, y_res_encoded = sm.fit_resample(X_train_flat, y_train_seq)
else:
    print("One of your classes has only 1 sample. SMOTE requires at least 2.")
    print("Using RandomOverSampler")
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=20)
    X_res_flat, y_res_encoded = ros.fit_resample(X_train_flat, y_train_seq)

# RE-SHAPE TO 3D IMMEDIATELY
X_train_3D = X_res_flat.reshape(-1, seq_len, n_features).astype(np.float32)
X_val_3D = X_val_seq.astype(np.float32) # Val stays original
X_test_3D = X_test_seq.astype(np.float32)

# 4. Targeted Scaling (Fit on Train, Transform all)
scaler = StandardScaler()

num_indices = config['num_idx']

def scale_3d(data, indices, scaler_obj):
    # Only perform math if we actually have indices to scale
    if len(indices) > 0:
        reshaped = data[:, :, indices].reshape(-1, len(indices))
        data[:, :, indices] = scaler_obj.transform(reshaped).reshape(-1, seq_len, len(indices))
    return data

if(len(num_indices)>0):
    num_train_part = X_train_3D[:, :, num_indices].reshape(-1, len(num_indices))
    scaler.fit(num_train_part)

    X_train_3D = scale_3d(X_train_3D, num_indices, scaler)
    X_val_3D = scale_3d(X_val_3D, num_indices, scaler)
    X_test_3D = scale_3d(X_test_3D, num_indices, scaler)

# Helper to scale 3D volumes
# def scale_3d(data, indices, scaler):
#     reshaped = data[:, :, indices].reshape(-1, len(indices))
#     data[:, :, indices] = scaler.transform(reshaped).reshape(-1, seq_len, len(indices))
#     return data



# 5. Create Tensors and DataLoaders
X_train_tensor = torch.FloatTensor(X_train_3D)
y_train_tensor = torch.LongTensor(y_res_encoded)

X_val_tensor = torch.FloatTensor(X_val_3D)
y_val_tensor = torch.LongTensor(y_val_seq)

X_test_tensor = torch.FloatTensor(X_test_3D)
y_test_tensor = torch.LongTensor(y_test_seq)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print(f"Train: {X_train_tensor.shape} | Val: {X_val_tensor.shape} | Test: {X_test_tensor.shape}")

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


StandardScaler()

Train: torch.Size([67136, 64, 9]) | Val: torch.Size([2437, 64, 9]) | Test: torch.Size([8127, 64, 9])


In [62]:
# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Informer"

# 2. Initialize the Universal Informer Model
# Ensure 'config' is defined as we discussed (containing num_idx, cat_idx, etc.)
num_classes = len(np.unique(y_res_encoded)) # Dynamic class count from SMOTE labels
seq_len = 64
d_model=128
nhead=8
num_layers=3
mask=''
threshold=2.5

In [63]:
if(model_name=="Informer"):
    model = UniversalInformer(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,           # Transformer layers
    dropout=0.1).to(device)

if(model_name=="InformerMask"):
    model = UniversalInformerMask(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,          # Transformer layers
    masking_method=mask,
    threshold=threshold,
    dropout=0.1).to(device)

if(model_name=="CNNInformerMask"):
    model = UniversalInformerCNN(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,           # Transformer layers
    masking_method=mask,
    threshold=threshold,
    dropout=0.1).to(device)

In [64]:

print(f"Universal Informer initialized for {device}.")
print(f"Processing: {len(config['num_idx'])} continuous, "
      f"{len(config['cat_idx'])} categorical, and "
      f"{len(config['time_idx'])} temporal features.")

# 3. Define Loss, Optimizer, and Scheduler
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Adaptive learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=2, 
    factor=0.5, 
    verbose=True
)

Universal Informer initialized for cuda.
Processing: 3 continuous, 0 categorical, and 6 temporal features.


/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


## Training

In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(class_names)
model = UniversalInformer(config, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

print(f"Training on {device}...")

# Computational Measurement Start
start_stats_train = get_system_stats(device, reset=True)
start_time_train = time.time()


# Create a list of all possible label indices [0, 1, 2, ..., num_classes-1]
all_labels = list(range(len(class_names)))

for epoch in range(20):
    _=model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        _=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # --- Validation Phase ---
    _=model.eval()
    val_loss, val_preds, val_true, val_probs = 0, [], [], []
    with torch.no_grad():
        for v_X, v_y in val_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            val_loss += criterion(v_out, v_y).item()
            val_probs.extend(torch.softmax(v_out, dim=1).cpu().numpy())
            val_preds.extend(torch.max(v_out, 1)[1].cpu().numpy())
            val_true.extend(v_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    # --- Metrics Calculation ---
    acc = accuracy_score(val_true, val_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(val_true, val_preds, average='weighted', zero_division=0)
    
    # Safe AUROC Calculation
    try:
        auroc = roc_auc_score(
            val_true, 
            val_probs, 
            multi_class='ovr', 
            average='weighted', 
            labels=all_labels  # Crucial: forces alignment with class_names
        )
    except ValueError:
        # Fallback if the batch is truly degenerate (e.g., only 1 class present)
        auroc = 0.0

    print(f"Epoch [{epoch+1:02d}/20] | Loss: {avg_val_loss:.4f} | Acc: {acc:.2f}% | F1: {f1:.4f} | AUROC: {auroc:.4f}")

# End Measurements
train_duration = time.time() - start_time_train
end_stats_train = get_system_stats(device)

Training on cuda...
Epoch [01/20] | Loss: 0.0790 | Acc: 97.99% | F1: 0.9807 | AUROC: 0.9950
Epoch [02/20] | Loss: 0.0393 | Acc: 99.10% | F1: 0.9912 | AUROC: 0.9983
Epoch [03/20] | Loss: 0.0572 | Acc: 98.93% | F1: 0.9894 | AUROC: 0.9982
Epoch [04/20] | Loss: 0.0775 | Acc: 98.48% | F1: 0.9852 | AUROC: 0.9966
Epoch [05/20] | Loss: 0.0539 | Acc: 98.77% | F1: 0.9880 | AUROC: 0.9982
Epoch [06/20] | Loss: 0.0557 | Acc: 98.89% | F1: 0.9890 | AUROC: 0.9986
Epoch [07/20] | Loss: 0.0480 | Acc: 98.97% | F1: 0.9899 | AUROC: 0.9986
Epoch [08/20] | Loss: 0.0538 | Acc: 99.06% | F1: 0.9906 | AUROC: 0.9986
Epoch [09/20] | Loss: 0.0478 | Acc: 99.14% | F1: 0.9914 | AUROC: 0.9986
Epoch [10/20] | Loss: 0.0435 | Acc: 99.22% | F1: 0.9920 | AUROC: 0.9989
Epoch [11/20] | Loss: 0.0429 | Acc: 99.14% | F1: 0.9915 | AUROC: 0.9988
Epoch [12/20] | Loss: 0.0479 | Acc: 99.14% | F1: 0.9914 | AUROC: 0.9988
Epoch [13/20] | Loss: 0.0487 | Acc: 99.10% | F1: 0.9910 | AUROC: 0.9987
Epoch [14/20] | Loss: 0.0532 | Acc: 99.10% |

In [66]:

X_val.shape
X_val_scaled.shape
X_val_seq.shape
X_val_3D.shape
X_val_tensor.shape
np.shape(val_true)
np.shape(val_preds)

(78029, 9)

(78029, 9)

(2437, 64, 9)

(2437, 64, 9)

torch.Size([2437, 64, 9])

(2437,)

(2437,)

In [67]:
np.unique_counts(val_true)
np.unique_counts(val_preds)
class_names

UniqueCountsResult(values=array([0, 1, 2, 3, 4, 5, 6, 7]), counts=array([2100,  132,   57,   37,   93,   11,    3,    4]))

UniqueCountsResult(values=array([0, 1, 2, 3, 4, 5, 6, 7]), counts=array([2089,  138,   62,   38,   93,   11,    2,    4]))

['normal',
 'backdoor',
 'ddos',
 'injection',
 'password',
 'ransomware',
 'scanning',
 'xss']

In [68]:
# 1. Generate the report dictionary
df_metrics_train = get_classification_df(val_true, 
                                         val_preds,
                                         val_probs,
                                         class_names=class_names,
                                         sensor_name=sensor_name, 
                                         model_name=model_name+mask, 
                                         phase="Training")
df_metrics_train

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.998085,0.992857,0.995464,2100.000000,Informer,Weather,Training
1,backdoor,0.956522,1.000000,0.977778,132.000000,Informer,Weather,Training
2,ddos,0.919355,1.000000,0.957983,57.000000,Informer,Weather,Training
3,injection,0.973684,1.000000,0.986667,37.000000,Informer,Weather,Training
4,password,0.956989,0.956989,0.956989,93.000000,Informer,Weather,Training
5,ransomware,1.000000,1.000000,1.000000,11.000000,Informer,Weather,Training
6,scanning,1.000000,0.666667,0.800000,3.000000,Informer,Weather,Training
7,xss,0.750000,0.750000,0.750000,4.000000,Informer,Weather,Training
8,accuracy,0.991383,0.991383,0.991383,0.991383,Informer,Weather,Training
9,macro avg,0.944329,0.920814,0.928110,2437.000000,Informer,Weather,Training


In [69]:
df_perf_train = get_performance_df(
    model_name=model_name+mask,
    sensor_name=sensor_name,
    phase='Training',
    duration=train_duration,
    stats=end_stats_train
)

df_perf_train

,Metric,Value,Unit,model,sensor,phase
0,Duration,94.7352,Seconds,Informer,Weather,Training
1,Peak CPU Memory,2284.0900,MB,Informer,Weather,Training
2,Avg CPU Utilization,8.1000,%,Informer,Weather,Training
3,Peak GPU Memory,149.7800,MB,Informer,Weather,Training
4,Avg GPU Utilization,0.0000,%,Informer,Weather,Training


In [70]:
# 1. Setup for Final Evaluation
print("\n" + "="*50)
print("FINAL TEST EVALUATION STARTED")
print("="*50)

# Load best model weights if you saved them during training
# model.load_state_dict(torch.load('best_model.pth')) 

_=model.eval()
test_loss, test_preds, test_true, test_probs = 0, [], [], []

# 2. Start Computational Performance Tracking
start_stats_test = get_system_stats(device, reset=True)
start_time_test = time.time()

# 3. Inference Loop
with torch.no_grad():
    for t_X, t_y in test_loader:
        t_X, t_y = t_X.to(device), t_y.to(device)
        
        # Forward pass
        t_out = model(t_X)
        
        # Calculate loss (optional for test set, but good for comparison)
        test_loss += criterion(t_out, t_y).item()
        
        # Store results
        test_probs.extend(F.softmax(t_out, dim=1).cpu().numpy())
        test_preds.extend(torch.max(t_out, 1)[1].cpu().numpy())
        test_true.extend(t_y.cpu().numpy())

# 4. End Computational Performance Tracking
test_duration = time.time() - start_time_test
end_stats_test = get_system_stats(device)
avg_test_loss = test_loss / len(test_loader)

# 5. Calculate Model Metrics
test_true = np.array(test_true)
test_probs = np.array(test_probs)

accuracy = accuracy_score(test_true, test_preds) * 100
# Handle AUROC for binary vs multi-class dynamically
if num_classes == 2:
    auroc = roc_auc_score(test_true, test_probs[:, 1])
else:
    auroc = roc_auc_score(test_true, test_probs, multi_class='ovr', average='weighted')

print(f"Final Test Accuracy: {accuracy:.2f}%")
print(f"Final Test AUROC: {auroc:.4f}")
print(f"Inference Time: {test_duration:.4f}s")


FINAL TEST EVALUATION STARTED
Final Test Accuracy: 99.21%
Final Test AUROC: 0.9981
Inference Time: 0.1656s


In [71]:
# 1. Generate the report dictionary
df_metrics_test = get_classification_df(test_true, test_preds, test_probs,
                                        class_names=class_names, 
                                        sensor_name=sensor_name, 
                                        model_name=model_name+mask, 
                                        phase="Testing")
df_metrics_test

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.997557,0.993559,0.995554,6986.000000,Informer,Weather,Testing
1,backdoor,0.971491,0.977925,0.974697,453.000000,Informer,Weather,Testing
2,ddos,0.930000,0.984127,0.956298,189.000000,Informer,Weather,Testing
3,injection,0.968254,1.000000,0.983871,122.000000,Informer,Weather,Testing
4,password,0.975535,0.981538,0.978528,325.000000,Informer,Weather,Testing
5,ransomware,0.923077,1.000000,0.960000,36.000000,Informer,Weather,Testing
6,scanning,0.555556,1.000000,0.714286,5.000000,Informer,Weather,Testing
7,xss,0.916667,1.000000,0.956522,11.000000,Informer,Weather,Testing
8,accuracy,0.992125,0.992125,0.992125,0.992125,Informer,Weather,Testing
9,macro avg,0.904767,0.992144,0.939969,8127.000000,Informer,Weather,Testing


In [72]:
df_perf_test = get_performance_df(
    model_name=model_name+mask,
    sensor_name=sensor_name,
    phase='Testing',
    duration=test_duration,
    stats=end_stats_test
)

df_perf_test

,Metric,Value,Unit,model,sensor,phase
0,Duration,0.1656,Seconds,Informer,Weather,Testing
1,Peak CPU Memory,2284.0900,MB,Informer,Weather,Testing
2,Avg CPU Utilization,7.0000,%,Informer,Weather,Testing
3,Peak GPU Memory,45.1900,MB,Informer,Weather,Testing
4,Avg GPU Utilization,0.0000,%,Informer,Weather,Testing


In [73]:
# Final Reports
# Classification
df_metrics = pd.concat([df_metrics_train, df_metrics_test], axis=0)
df_metrics

# Classification
df_perf = pd.concat([df_perf_train, df_perf_test], axis=0)
df_perf

#Saving Classification Report
save_report(df=df_metrics, report_type="classification",output_dir="../output/")

save_report(df=df_perf, report_type="performance",output_dir="../output/")

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.998085,0.992857,0.995464,2100.000000,Informer,Weather,Training
1,backdoor,0.956522,1.000000,0.977778,132.000000,Informer,Weather,Training
2,ddos,0.919355,1.000000,0.957983,57.000000,Informer,Weather,Training
3,injection,0.973684,1.000000,0.986667,37.000000,Informer,Weather,Training
4,password,0.956989,0.956989,0.956989,93.000000,Informer,Weather,Training
5,ransomware,1.000000,1.000000,1.000000,11.000000,Informer,Weather,Training
6,scanning,1.000000,0.666667,0.800000,3.000000,Informer,Weather,Training
7,xss,0.750000,0.750000,0.750000,4.000000,Informer,Weather,Training
8,accuracy,0.991383,0.991383,0.991383,0.991383,Informer,Weather,Training
9,macro avg,0.944329,0.920814,0.928110,2437.000000,Informer,Weather,Training


,Metric,Value,Unit,model,sensor,phase
0,Duration,94.7352,Seconds,Informer,Weather,Training
1,Peak CPU Memory,2284.0900,MB,Informer,Weather,Training
2,Avg CPU Utilization,8.1000,%,Informer,Weather,Training
3,Peak GPU Memory,149.7800,MB,Informer,Weather,Training
4,Avg GPU Utilization,0.0000,%,Informer,Weather,Training
0,Duration,0.1656,Seconds,Informer,Weather,Testing
1,Peak CPU Memory,2284.0900,MB,Informer,Weather,Testing
2,Avg CPU Utilization,7.0000,%,Informer,Weather,Testing
3,Peak GPU Memory,45.1900,MB,Informer,Weather,Testing
4,Avg GPU Utilization,0.0000,%,Informer,Weather,Testing


✅ Report successfully saved to: ../output/classification_report.xlsx | Sheet: Informer_Weather
✅ Report successfully saved to: ../output/computational_performance.xlsx | Sheet: Informer_Weather
